# The Simple Standard CEM Strategy — Polymarket → US-Equity Information-Diffusion Lag

This notebook presents and runs the project's **simple standard CEM strategy** — the
`Baseline` arm of the experiment matrix — **end-to-end and fully self-contained**: it
reads only the committed data artifacts and imports **no project code**. The polarity
resolver, the trade kernel, and the CEM optimizer are all re-implemented inline (§2), so
this single `.ipynb` is the complete strategy.

1. it documents the upstream data pipeline (documentation only; nothing is re-downloaded),
2. defines the self-contained computation engine — polarity + trade kernel + CEM search,
3. displays and exports the **complete** Polymarket-question → stock/ETF mapping,
4. runs the standard CEM fit + out-of-sample backtest twice — once against **SPY**,
   once against **QQQ** — using only the inlined engine,
5. plots the portfolio-vs-benchmark equity curves and writes all practical output files.

Everything runs **offline**: no Polymarket API calls, no Gemini calls, no database
connection, no subprocess, no import of `core/` or `backtesting/`. The only inputs are
the four committed artifacts under `data/`.

## 1. Requirements

| Package | Why it is needed |
|---|---|
| `numpy`, `pandas` | data handling and simulation math |
| `pyarrow` | reads `data/candidates_audit_clean.parquet` |
| `matplotlib` | equity-curve figures |
| `numba` | JIT-compiled fast path for the trade kernel (a pure-Python fallback exists, but the CEM search is much faster with numba installed) |
| `jupyter` | to run this notebook |

Install everything from the repository root with:

```bash
pip install -r requirements.txt
```

or directly:

```bash
pip install numpy pandas pyarrow matplotlib numba jupyter
```

Developed on Python 3.13; any Python ≥ 3.11 should work.

In [ ]:
import json
import math
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    """Locate the repository root (the directory holding the committed data artifacts)."""
    for p in (start, *start.parents):
        if (p / "data" / "candidates_audit_clean.parquet").exists():
            return p
    raise FileNotFoundError(
        "Could not locate data/candidates_audit_clean.parquet from the current directory."
    )


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data"

# The ONLY inputs to this notebook — four committed data artifacts. No project
# code is imported: the polarity resolver, the trade kernel, and the CEM search
# are all re-implemented in the cells below (§2).
CANDIDATES_PATH = DATA_DIR / "candidates_audit_clean.parquet"
PRICES_PATH = DATA_DIR / "prices.pkl"
PROBS_PATH = DATA_DIR / "probs.pkl"
POLARITY_PATH = DATA_DIR / "polarity_labels.json"

OUT_DIR = REPO_ROOT / "notebooks" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"repo root : {REPO_ROOT}")
print(f"data dir  : {DATA_DIR}")
print(f"outputs   : {OUT_DIR}")
print(f"python    : numpy {np.__version__}   pandas {pd.__version__}")
try:
    import numba
    print(f"numba     : {numba.__version__} (JIT kernel active)")
except ImportError:
    print("numba     : not installed — the kernel runs as pure Python (slower, identical results)")

## 2. The computation engine (self-contained)

Everything the standard CEM strategy needs is defined in the three code cells below —
**no project module is imported**. These are faithful, minimal re-implementations of the
research repository's logic, trimmed to exactly the **Baseline** arm (plain CEM: no T1
friction penalty, no T2 walk-forward refits, no T3 half-Kelly sizing, no T4 geo-priority
allocation, no PPO):

- **§2.1 Signal polarity** (from `core/polarity.py`) — resolves, per (question, symbol)
  pair, whether a rising P(YES) is bullish (`+1`), bearish (`−1`, path flipped to
  `1 − P(YES)`), or has no clean side (`0`, skipped). Resolution order: manual
  **override** → committed **LLM label** (`data/polarity_labels.json`) → **regex** fallback.
- **§2.2 The trade kernel** (from `core/kernel.py`) — the one authoritative entry/exit
  engine. `simulate_one` produces a single trade (entry rule, ATR trailing stop,
  profit-lock, probability-collapse exit, resolution/end-of-window liquidation). It is
  JIT-compiled by numba when available and runs as identical pure Python otherwise.
- **§2.3 Portfolio simulator + CEM search** (from `backtesting/optimize_cem.py`) — the
  benchmark-rotation portfolio (fully-invested, IB-style costs, FIFO allocation) and the
  cross-entropy-method optimizer that fits the policy on label-complete training data.

The numbers this notebook produces are byte-identical to the project's canonical seed-42
standard-CEM run (verified in §7).

In [ ]:
# ── 2.1  Signal polarity (inlined from core/polarity.py) ─────────────────────
# For each (question, symbol) pair: does a rising P(YES) help or hurt a LONG in
# the symbol? Resolution precedence: override > committed LLM label > regex.
RELEVANCE_COL = "feat_connection_strength"

# Regex fallback — each rule toggles the sign, so composition works.
_NEG_RE = re.compile(r"^\s*no\b|\bnot\b|\bnever\b|\bwithout\b|\brefrains?\s+from\b", re.I)
_CESSATION_RE = re.compile(
    r"\bends?\b(?!\s+of\b)|\bending\b(?!\s+of\b)"
    r"|\bceasefire\b|\bde-?escalat\w*|\bwithdraw\w*|\breturns?\s+to\s+normal\b", re.I)
_MACRO_UP_RE = re.compile(r"\babove\b|\bexceeds?\b|\bhikes?\b|\braises?\b", re.I)
_MACRO_SUBJ_RE = re.compile(r"\binflation\b|\bcpi\b|\brates?\b", re.I)
_SUPPLY_UP_RE = re.compile(r"\babove\b|\bexceeds?\b|\brises?\b|\bincreases?\b", re.I)
_SUPPLY_SUBJ_RE = re.compile(r"\bproduction\b|\boutput\b|\bsupply\b", re.I)
_COMMODITY_RE = re.compile(r"\bcrude\b|\boil\b|\bopec\b|\bbarrels?\b|\bnatural\s+gas\b|\bgas\b", re.I)
_BEARISH_EVENT_RE = re.compile(
    r"\bmiss(?:es|ed)?\b|\bfalls?\b|\bfallen\b|\bdeclin\w*|\bcrash\w*"
    r"|\bfails?\s+to\b|\brejects?\b|\bdowngrade\w*|\bbankrupt\w*"
    r"|\bdefaults?\b|\blayoffs?\b", re.I)

_POLARITY_RULES = (
    ("R1_negation", lambda q: bool(_NEG_RE.search(q))),
    ("R2_cessation", lambda q: bool(_CESSATION_RE.search(q))),
    ("R3_macro_level_up", lambda q: bool(_MACRO_UP_RE.search(q) and _MACRO_SUBJ_RE.search(q))),
    ("R4_commodity_supply_up",
     lambda q: bool(_SUPPLY_UP_RE.search(q) and _SUPPLY_SUBJ_RE.search(q) and _COMMODITY_RE.search(q))),
    ("R5_bearish_event", lambda q: bool(_BEARISH_EVENT_RE.search(q))),
)


def explain_polarity(question: str) -> tuple[int, list[str]]:
    """Regex-only polarity: (sign, names of rules that fired). Binary (never 0)."""
    q = (question or "").strip()
    fired = [name for name, test in _POLARITY_RULES if test(q)]
    return (-1 if len(fired) % 2 else 1), fired


# Hand-verified domain facts the LLM gets wrong: container carriers (ZIM) gain
# from port congestion, so a strike is bullish and its ending bearish.
POLARITY_OVERRIDES = {
    ("east coast port strike ends by friday?", "ZIM"): -1,
    ("east coast port strike ends by next friday?", "ZIM"): -1,
    ("east coast port strike ends in october?", "ZIM"): -1,
    ("longshoremen east coast strike by oct 1?", "ZIM"): +1,
}

_LLM_LABELS = None


def _norm(question: str, symbol: str) -> tuple[str, str]:
    return (question or "").strip().lower(), (symbol or "").strip().upper()


def _llm_labels() -> dict:
    """Load the committed per-pair LLM polarity labels (data/polarity_labels.json)."""
    global _LLM_LABELS
    if _LLM_LABELS is None:
        if POLARITY_PATH.exists():
            raw = json.loads(POLARITY_PATH.read_text(encoding="utf-8"))
            _LLM_LABELS = {_norm(rec["question"], rec["symbol"]): int(rec["polarity"])
                           for rec in raw.values()}
        else:
            _LLM_LABELS = {}
    return _LLM_LABELS


def resolve_polarity(question: str, symbol: str | None = None) -> tuple[int, str]:
    """Return (polarity, source) where source is override | llm | regex."""
    if symbol is not None:
        key = _norm(question, symbol)
        if key in POLARITY_OVERRIDES:
            return POLARITY_OVERRIDES[key], "override"
        label = _llm_labels().get(key)
        if label is not None:
            return label, "llm"
    return explain_polarity(question)[0], "regex"


def effective_prob_path(prob_path, polarity):
    """Re-polarize a probability path so that "high" always means "bullish"."""
    if polarity == 1:
        return prob_path
    if polarity == -1:
        return [(t, 1.0 - v) for t, v in prob_path]
    raise ValueError("polarity 0 has no tradable probability path")


# Memoize effective-probs dicts per (source id, polarity) so the numba kernel's
# (id(probs), mkt) cache keeps hitting; hold a strong ref so the id can't recycle.
_EFF_PROBS_CACHE: dict = {}


def effective_probs(probs, mkt, polarity):
    if polarity not in {-1, 1}:
        raise ValueError("polarity 0 has no tradable probability path")
    key = (id(probs), polarity)
    cached = _EFF_PROBS_CACHE.get(key)
    if cached is None or cached[0] is not probs:
        cached = (probs, {})
        _EFF_PROBS_CACHE[key] = cached
    _source, eff = cached
    if mkt not in eff:
        eff[mkt] = effective_prob_path(probs.get(mkt, []), polarity)
    return eff


def effective_prob_surge(row, polarity):
    """feat_prob_surge_since_t0 is a delta on the raw path, so it flips too."""
    surge = row.get("feat_prob_surge_since_t0")
    if polarity == 1 or surge is None:
        return surge
    try:
        value = float(surge)
    except (TypeError, ValueError):
        return surge
    return surge if value != value else -value  # NaN passes through unchanged


def clear_effective_probs_cache():
    _EFF_PROBS_CACHE.clear()


print("polarity engine ready (override > llm > regex)")

In [ ]:
# ── 2.2  The trade kernel (inlined from core/kernel.py) ──────────────────────
# The one authoritative entry/exit engine. `_scan` is the numeric core; numba
# JIT-compiles it when available and it runs as identical pure Python otherwise.
# Long-only: `core.polarity` decides per pair whether the effective path is raw
# P(YES) (+1), the flipped 1-P(YES) (-1), or no clean side (0 -> skipped).
try:
    from numba import njit
    HAVE_NUMBA = True
except Exception:  # numba is optional; the same code runs as pure Python.
    HAVE_NUMBA = False

    def njit(*args, **kwargs):
        if args and callable(args[0]) and len(args) == 1 and not kwargs:
            return args[0]

        def _wrap(fn):
            return fn
        return _wrap


_DAY_NS = 86_400_000_000_000
_SYM_CACHE: dict = {}   # (id(prices), sym) -> arrays
_MKT_CACHE: dict = {}   # (id(probs), mkt) -> arrays


def clear_caches():
    _SYM_CACHE.clear()
    _MKT_CACHE.clear()


def clear_kernel_caches():
    """Drop cached numpy views AND the polarity-corrected probability paths."""
    clear_caches()
    clear_effective_probs_cache()


def _symbol_arrays(prices, sym):
    """Cached (value, norm, high, low, close, bars) int64/float64 arrays for sym."""
    key = (id(prices), sym)
    cached = _SYM_CACHE.get(key)
    if cached is not None:
        return cached
    bars = prices.get(sym, [])
    n = len(bars)
    value = np.empty(n, dtype=np.int64)
    norm = np.empty(n, dtype=np.int64)
    high = np.empty(n, dtype=np.float64)
    low = np.empty(n, dtype=np.float64)
    close = np.empty(n, dtype=np.float64)
    for i in range(n):
        bar = bars[i]
        ts = bar[0] if isinstance(bar[0], pd.Timestamp) else pd.Timestamp(bar[0])
        value[i] = ts.value
        norm[i] = ts.normalize().value
        high[i] = bar[1]
        low[i] = bar[2]
        close[i] = bar[3]
    cached = (value, norm, high, low, close, bars)
    _SYM_CACHE[key] = cached
    return cached


def _market_arrays(probs, mkt):
    """Cached (pt_value, pval_raw, day_uni, pval_uni, points) arrays for a market."""
    key = (id(probs), mkt)
    cached = _MKT_CACHE.get(key)
    if cached is not None:
        return cached
    points = probs.get(mkt, [])
    m = len(points)
    pt_value = np.empty(m, dtype=np.int64)
    pval_raw = np.empty(m, dtype=np.float64)
    day_to_val: dict = {}
    for i in range(m):
        point = points[i]
        ts = point[0] if isinstance(point[0], pd.Timestamp) else pd.Timestamp(point[0])
        pt_value[i] = ts.value
        value = float(point[1])
        pval_raw[i] = value
        day_to_val[ts.normalize().value] = value  # last point of a day wins
    if day_to_val:
        day_uni = np.array(sorted(day_to_val.keys()), dtype=np.int64)
        pval_uni = np.array([day_to_val[d] for d in day_uni], dtype=np.float64)
    else:
        day_uni = np.empty(0, dtype=np.int64)
        pval_uni = np.empty(0, dtype=np.float64)
    cached = (pt_value, pval_raw, day_uni, pval_uni, points)
    _MKT_CACHE[key] = cached
    return cached


@njit(cache=True)
def _bisect_left(a, x):
    lo, hi = 0, a.shape[0]
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo


@njit(cache=True)
def _bisect_right(a, x):
    lo, hi = 0, a.shape[0]
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] <= x:
            lo = mid + 1
        else:
            hi = mid
    return lo


# Reason codes: 0 none, 1 trailing-ATR, 2 profit-lock, 3 probability-out,
# 4 resolution-1d, 5 end-of-window.
@njit(cache=True)
def _scan(bar_value, bar_norm, bar_high, bar_low, bar_close,
          pt_value, pval_raw, day_uni, pval_uni,
          window_lo_value, t_e_value, first_eligible_value, resolution_cut_value,
          is_earnings, enter_strong, enter_floor, hold_days, atr_mult, lock_activate,
          theta_out, p_surge, max_prob_surge, r_surge, max_price_runup):
    none = (0, -1, -1, -1, 0.0, 0, 0, 0.0, 0.0, 0.0, 0.0)
    w_start = _bisect_left(bar_value, window_lo_value)
    w_end = _bisect_right(bar_value, t_e_value)
    if w_end - w_start < 2:
        return none
    m = pt_value.shape[0]
    e0 = _bisect_left(pt_value, first_eligible_value)
    if e0 >= m:
        return none
    entry_pt_index = -1
    held = 0
    k = e0
    while k < m:
        if pval_raw[k] >= enter_strong:
            entry_pt_index = k
            break
        elif pval_raw[k] >= enter_floor:
            held += 1
            if held >= hold_days:
                entry_pt_index = k
                break
        else:
            held = 0
        k += 1
    if entry_pt_index < 0:
        return none
    entry_ts_value = pt_value[entry_pt_index]
    if (p_surge == p_surge) and (p_surge > max_prob_surge):
        return none
    if (r_surge == r_surge) and (r_surge > max_price_runup):
        return none
    gi = _bisect_left(bar_value, entry_ts_value)
    if gi < w_start:
        gi = w_start
    if gi >= w_end:
        return none
    if w_end - gi < 2:
        return none
    if bar_value[gi] >= resolution_cut_value:
        return none
    hold_end = _bisect_left(bar_value, resolution_cut_value)
    if hold_end > w_end:
        hold_end = w_end
    if hold_end - gi < 2:
        return none
    entry_price = bar_close[gi]
    h_start = gi - 15
    if h_start < w_start:
        h_start = w_start
    tr_sum = 0.0
    cnt = 0
    j = h_start + 1
    while j <= gi:
        hh = bar_high[j]
        ll = bar_low[j]
        pc = bar_close[j - 1]
        tr = hh - ll
        d2 = hh - pc
        if d2 < 0.0:
            d2 = -d2
        if d2 > tr:
            tr = d2
        d3 = ll - pc
        if d3 < 0.0:
            d3 = -d3
        if d3 > tr:
            tr = d3
        tr_sum += tr
        cnt += 1
        j += 1
    if cnt < 1:
        return none
    atr = tr_sum / cnt
    if atr == 0.0 or entry_price == 0.0:
        return none
    atr_pct = atr / entry_price
    peak = 0.0
    gj = gi
    while gj < hold_end:
        i_rel = gj - gi
        hh = bar_high[gj]
        ll = bar_low[gj]
        cc = bar_close[gj]
        ret_c = cc / entry_price - 1.0
        ret_h = hh / entry_price - 1.0
        ret_l = ll / entry_price - 1.0
        reason = 0
        hard_floor_pct = 0
        if i_rel > 0:
            stop_dist = atr_mult * atr_pct
            pv = 1.0
            idx = _bisect_left(day_uni, bar_norm[gj])
            if idx < day_uni.shape[0] and day_uni[idx] == bar_norm[gj]:
                pv = pval_uni[idx]
            if pv < theta_out:
                reason = 3
            elif ret_l <= peak - stop_dist:
                reason = 1
                cand = entry_price * (1.0 + peak - stop_dist)
                cc = ll if ll > cand else cand
                ret_c = cc / entry_price - 1.0
            elif peak >= lock_activate:
                hard_floor_pct = int(peak * 100.0)
                hard_floor = hard_floor_pct / 100.0
                if ret_l < hard_floor:
                    reason = 2
                    cand = entry_price * (1.0 + hard_floor)
                    cc = ll if ll > cand else cand
                    ret_c = cc / entry_price - 1.0
            if reason == 0 and gj == hold_end - 1:
                reason = 4
        if reason != 0:
            lo = 0.0
            first = 1
            k = gi
            while k <= gj:
                rl = bar_low[k] / entry_price - 1.0
                if first == 1 or rl < lo:
                    lo = rl
                    first = 0
                k += 1
            return (1, int(entry_pt_index), int(gi), int(gj), cc, int(reason),
                    int(hard_floor_pct), peak, lo, ret_c, entry_price)
        if i_rel == 0:
            peak = 0.0
        elif ret_h > peak:
            peak = ret_h
        gj += 1
    last = hold_end - 1
    cc = bar_close[last]
    ret_c = cc / entry_price - 1.0
    lo = 0.0
    first = 1
    k = gi
    while k < hold_end:
        rl = bar_low[k] / entry_price - 1.0
        if first == 1 or rl < lo:
            lo = rl
            first = 0
        k += 1
    return (1, int(entry_pt_index), int(gi), int(last), cc, 5, 0, peak, lo, ret_c, entry_price)


def scan_candidate(prices, probs, sym, mkt, t_theta, t_e, is_earnings, p_surge, r_surge, policy):
    """Run the kernel for one candidate; return None or a formatted trade tuple."""
    bar_value, bar_norm, bar_high, bar_low, bar_close, bars = _symbol_arrays(prices, sym)
    if bar_value.shape[0] < 2:
        return None
    pt_value, pval_raw, day_uni, pval_uni, points = _market_arrays(probs, mkt)
    if pt_value.shape[0] == 0:
        return None
    window_lo_value = np.int64(t_theta.value) - 30 * _DAY_NS
    t_e_value = np.int64(t_e.value)
    first_eligible_value = np.int64(t_theta.normalize().value)
    resolution_cut_value = np.int64((t_e - pd.Timedelta(days=1)).value)
    surge = float(p_surge) if p_surge is not None else float("nan")
    runup = float(r_surge) if r_surge is not None else float("nan")
    result = _scan(bar_value, bar_norm, bar_high, bar_low, bar_close,
                   pt_value, pval_raw, day_uni, pval_uni,
                   window_lo_value, t_e_value, first_eligible_value, resolution_cut_value,
                   1 if is_earnings else 0,
                   float(policy["enter_strong"]), float(policy["enter_floor"]), int(policy["hold_days"]),
                   float(policy["atr_mult"]), float(policy["lock_activate"]), float(policy["theta_out"]),
                   surge, float(policy.get("max_prob_surge", 999.0)),
                   runup, float(policy.get("max_price_runup", 999.0)))
    if result[0] == 0:
        return None
    (_status, entry_pt_index, _entry_gindex, exit_gindex, exit_price, reason_code,
     hard_floor_pct, peak, trough, ret_c, entry_price) = result
    entry_point = points[int(entry_pt_index)]
    exit_bar = bars[int(exit_gindex)]
    return (bars[int(_entry_gindex)][0], float(entry_point[1]), float(entry_price),
            exit_bar[0], float(exit_price), int(reason_code), int(hard_floor_pct),
            float(peak), float(trough), float(ret_c))


def simulate_one(row, prices, probs, policy):
    """Simulate a single trade. Returns a trade dict or None."""
    sym, mkt = row["symbol"], row["market_id"]
    # Resolve polarity here: a bearish-YES pair is flipped (never shorted), a
    # no-clean-side pair (polarity 0) is skipped.
    question = str(row.get("question", ""))
    polarity, polarity_source = resolve_polarity(question, sym)
    if polarity == 0:
        return None
    probs = effective_probs(probs, mkt, polarity)
    t_theta = pd.Timestamp(row["t_theta"]).tz_convert("UTC")
    t_e = pd.Timestamp(row["t_e"]).tz_convert("UTC")
    is_earnings = "earnings" in str(row.get("feat_archetype", "")).lower()
    scanned = scan_candidate(prices, probs, sym, mkt, t_theta, t_e, is_earnings,
                             effective_prob_surge(row, polarity),
                             row.get("feat_runup_since_t0"), policy)
    if scanned is None:
        return None
    (entry_ts, entry_prob, entry_price, exit_ts, exit_price, reason_code,
     hard_floor_pct, peak, trough, ret_c) = scanned
    if reason_code == 1:
        reason = f"trailing_{policy['atr_mult']:.1f}ATR"
    elif reason_code == 2:
        reason = f"profit_lock_{hard_floor_pct}%"
    elif reason_code == 3:
        reason = f"poly<{policy['theta_out']}"
    elif reason_code == 4:
        reason = "resolution-1d"
    else:
        reason = "end_of_window"
    mkt_probs = probs.get(mkt, [])
    converged = "YES" if mkt_probs and mkt_probs[-1][1] >= 0.5 else "NO" if mkt_probs else "UNKNOWN"
    return dict(
        market_id=mkt, symbol=sym, question=question,
        polarity=polarity, polarity_source=polarity_source,
        pct=round(entry_prob, 3), converged=converged,
        asset_confidence=row.get("confidence_score"),
        question_confidence=row.get("feat_llm_confidence"),
        archetype=row.get("feat_archetype", ""),
        relevance=round(float(row.get(RELEVANCE_COL, 0)), 3),
        split=row.get("split", ""),
        entry_date=str(entry_ts.date()), entry_prob=round(entry_prob, 3),
        entry_price=round(entry_price, 2),
        exit_date=str(exit_ts.date()), exit_price=round(exit_price, 2),
        exit_reason=reason,
        peak_pct=round(peak * 100, 2), trough_pct=round(trough * 100, 2),
        return_pct=round(ret_c * 100, 2),
    )


print(f"trade kernel ready (numba {'active' if HAVE_NUMBA else 'absent — pure-Python path'})")

In [ ]:
# ── 2.3  Portfolio simulator + CEM (inlined from backtesting/optimize_cem.py) ─
# Baseline arm only: fully-invested benchmark-rotation portfolio, IB-style costs,
# FIFO allocation, fixed position_size_pct (no Kelly), single CEM fit (no folds),
# plain Sharpe−λ·|maxDD| objective (no friction hurdle).
INITIAL_CAPITAL = 100_000.0
FULLY_INVESTED_SWEEP = True     # idle cash is swept into the benchmark daily
FRACTIONAL_BENCHMARK = True     # SPY/QQQ rotation legs may use fractional shares
MIN_SWEEP_CASH = 100.0          # don't churn sub-$100 sweeps

DEFAULT_POLICY = dict(
    atr_mult=3.65, lock_activate=0.03, theta_out=0.55, enter_strong=0.75,
    enter_floor=0.70, hold_days=1, max_prob_surge=0.40, max_price_runup=0.10,
)
PORTFOLIO_BOUNDS = dict(
    atr_mult=(1.5, 4.0), lock_activate=(0.02, 0.10), theta_out=(0.45, 0.60),
    enter_strong=(0.60, 0.85), enter_floor=(0.55, 0.80), hold_days=(1, 5),
    max_prob_surge=(0.20, 0.80), max_price_runup=(0.02, 0.20),
    position_size_pct=(0.06, 0.12), max_concurrent=(8, 12),
)
PORT_DEFAULT = {**DEFAULT_POLICY, "position_size_pct": 0.10, "max_concurrent": 10}

CEM_ITERS = 6
CEM_POP = 20
CEM_ELITE_FRAC = 0.25
CEM_BASE_SEED = 42
BENCHMARK_SEED_OFFSET = {"SPY": 0, "QQQ": 10_000}
MIN_TRADES_FOR_REWARD = 3
MIN_DAILY_RETURNS_FOR_SHARPE = 20
DD_PENALTY = 0.30
INVALID_SCORE = -1e9

_CLOSE_CACHE: dict = {}
_PATH_CUTOFF_CACHE: dict = {}


def as_utc_day(value):
    ts = pd.Timestamp(value)
    ts = ts.tz_localize("UTC") if ts.tz is None else ts.tz_convert("UTC")
    return ts.normalize()


def _as_utc_timestamp(value):
    ts = pd.Timestamp(value)
    return ts.tz_localize("UTC") if ts.tz is None else ts.tz_convert("UTC")


def _frame_bounds(df):
    ts = pd.to_datetime(df["t_theta"], utc=True)
    return as_utc_day(ts.min()), as_utc_day(ts.max())


def rows_completed_before(df, cutoff):
    """Rows legally available for fitting at `cutoff` (t_e < cutoff, strict)."""
    cutoff_ts = _as_utc_timestamp(cutoff)
    completed_at = pd.to_datetime(df["t_e"], utc=True)
    return df.loc[completed_at < cutoff_ts].copy()


def assert_rows_completed_before(df, cutoff, *, context):
    """Fail closed if a CEM fit contains any label incomplete at its cutoff."""
    if df.empty:
        raise ValueError(f"{context}: no rows are available for fitting.")
    cutoff_ts = _as_utc_timestamp(cutoff)
    if (pd.to_datetime(df["t_e"], utc=True) >= cutoff_ts).any():
        raise ValueError(f"{context}: fit rows have t_e >= {cutoff_ts.isoformat()}.")


def ib_cost(shares, price, is_sell):
    """IB-style commission + SEC fee on sales + fixed 5 bp slippage."""
    if shares <= 0 or price <= 0:
        return 0.0
    trade_value = shares * price
    commission = max(0.35, min(shares * 0.0035, trade_value * 0.01))
    sec = trade_value * 0.0000278 if is_sell else 0.0
    return commission + sec + trade_value * 0.0005


def _close_on(prices, symbol, date):
    """Latest known daily close on or before `date`, cached for speed."""
    key = (id(prices), symbol)
    cached = _CLOSE_CACHE.get(key)
    if cached is None:
        bars = prices.get(symbol, [])
        if not bars:
            return None
        days = np.array([as_utc_day(t).value for t, *_ in bars], dtype=np.int64)
        values = np.asarray([float(close) for *_rest, close in bars], dtype=float)
        cached = (days, values)
        _CLOSE_CACHE[key] = cached
    days, values = cached
    if type(date) is pd.Timestamp and date.tzinfo is not None and (date.value % _DAY_NS) == 0:
        date_ns = date.value
    else:
        date_ns = as_utc_day(date).value
    loc = int(np.searchsorted(days, date_ns, side="right")) - 1
    return None if loc < 0 else float(values[loc])


def _calendar_dates(prices, bench_sym, start_date, end_date):
    start, end = as_utc_day(start_date), as_utc_day(end_date)
    dates = sorted({as_utc_day(t) for t, *_ in prices.get(bench_sym, [])})
    return [d for d in dates if start <= d <= end]


def truncate_paths(prices, probs, end_date):
    """Return price/probability paths clipped to an evaluation horizon."""
    if end_date is None:
        return prices, probs
    cutoff = as_utc_day(end_date)
    key = (id(prices), id(probs), str(cutoff.date()))
    cached = _PATH_CUTOFF_CACHE.get(key)
    if cached is not None:
        return cached
    tp = {s: [b for b in bars if as_utc_day(b[0]) <= cutoff] for s, bars in prices.items()}
    tq = {m: [p for p in pts if as_utc_day(p[0]) <= cutoff] for m, pts in probs.items()}
    cached = (tp, tq)
    _PATH_CUTOFF_CACHE[key] = cached
    return cached


def _affordable_buy_qty(cash_available, price):
    """Largest integer share count whose buy price plus cost fits cash."""
    if cash_available <= 0 or price <= 0:
        return 0
    qty = int(cash_available / price)
    while qty > 0 and qty * price + ib_cost(qty, price, False) > cash_available + 1e-9:
        qty -= 1
    return qty


def _bench_buy_qty(cash_available, price):
    """Benchmark buy size: fractional shares when enabled, else whole shares."""
    if not FRACTIONAL_BENCHMARK:
        return float(_affordable_buy_qty(cash_available, price))
    if cash_available <= 0 or price <= 0:
        return 0.0
    qty = cash_available / price
    for _ in range(4):
        qty = max((cash_available - ib_cost(qty, price, False)) / price, 0.0)
    return qty


def port_policy_from_vec(vec):
    """Clip a CEM sample vector into a valid policy dict."""
    names = list(PORTFOLIO_BOUNDS.keys())
    policy = {}
    for i, name in enumerate(names):
        lo, hi = PORTFOLIO_BOUNDS[name]
        policy[name] = float(np.clip(vec[i], lo, hi))
    policy["hold_days"] = int(round(float(policy["hold_days"])))
    policy["max_concurrent"] = int(round(float(policy["max_concurrent"])))
    if float(policy["enter_strong"]) < float(policy["enter_floor"]):
        policy["enter_strong"] = policy["enter_floor"]
    return policy


def _calc_advanced_metrics(equity_series):
    if len(equity_series) < 2:
        return {"sharpe": 0.0, "sortino": 0.0}
    r = equity_series.pct_change().dropna()
    mu, sig = r.mean(), r.std()
    sharpe = (mu / sig * np.sqrt(252)) if sig > 1e-9 else 0.0
    down = r[r < 0]
    down_sig = down.std() if len(down) > 0 else 1e-9
    sortino = (mu / down_sig * np.sqrt(252)) if down_sig > 1e-9 else 0.0
    return {"sharpe": float(sharpe), "sortino": float(sortino)}


def sim_opp_cost(df, prices, probs, policy, *, bench_sym="SPY", initial=INITIAL_CAPITAL,
                 start_date=None, end_date=None):
    """Simulate a fully-invested benchmark-rotation portfolio (Baseline / FIFO).

    Capital starts fully in the benchmark; funding an event position sells
    benchmark shares and closing it re-buys them, so every round trip pays four
    IB-style cost legs. When `end_date` is supplied, open positions are
    liquidated at that date's close (leakage-safe train windows). Returns
    (trade_df, equity_df, stats).
    """
    empty_stats = {
        "initial": initial, "final": initial, "total_return": 0.0, "benchmark_return": 0.0,
        "excess_return": 0.0, "max_dd": 0.0, "sharpe": 0.0, "sortino": 0.0,
        "benchmark_sharpe": 0.0, "benchmark_sortino": 0.0, "n_trades": 0, "win_rate": 0.0,
        "avg_pnl": 0.0, "avg_gross_pnl": 0.0, "gross_trade_pnl": 0.0, "net_trade_pnl": 0.0,
        "total_txn_cost": 0.0, "trade_txn_cost": 0.0, "avg_position_size": 0.0,
        "median_position_size": 0.0, "min_position_size": 0.0, "max_position_size": 0.0,
        "start_date": None, "end_date": None, "n_equity_days": 0,
        "skip_max_concurrent": 0, "skip_duplicate_symbol": 0, "skip_insufficient_capital": 0,
    }
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), empty_stats

    sim_prices, sim_probs = truncate_paths(prices, probs, end_date)

    all_trades = []
    for _, row in df.sort_values("t_theta").iterrows():
        candidate_theta = as_utc_day(row["t_theta"])
        trade = simulate_one(row, sim_prices, sim_probs, policy)
        if trade is None:
            continue
        trade = dict(trade)
        trade["_entry_ts"] = as_utc_day(trade["entry_date"])
        trade["_exit_ts"] = as_utc_day(trade["exit_date"])
        if trade["_entry_ts"] < candidate_theta:
            raise ValueError(f"{trade['symbol']} entered before candidate t_theta.")
        trade["candidate_t_theta"] = str(candidate_theta.date())
        trade["candidate_t_e"] = str(as_utc_day(row["t_e"]).date())
        all_trades.append(trade)
    all_trades.sort(key=lambda t: t["_entry_ts"])

    candidate_start, candidate_end = _frame_bounds(df)
    eval_start = as_utc_day(start_date) if start_date is not None else min(
        (t["_entry_ts"] for t in all_trades), default=candidate_start)
    eval_end = as_utc_day(end_date) if end_date is not None else max(
        (t["_exit_ts"] for t in all_trades), default=candidate_end)
    if end_date is not None and [t for t in all_trades if t["_exit_ts"] > eval_end]:
        raise ValueError("generated trades exit after evaluation end.")

    if not sim_prices.get(bench_sym, []):
        raise ValueError(f"No daily bars available for benchmark {bench_sym}.")
    calendar = _calendar_dates(sim_prices, bench_sym, eval_start, eval_end)
    if not calendar:
        raise ValueError("No benchmark bars overlap evaluation range.")
    first_day, last_day = calendar[0], calendar[-1]
    first_bench_close = _close_on(sim_prices, bench_sym, first_day)
    last_bench_close = _close_on(sim_prices, bench_sym, last_day)
    if first_bench_close is None or last_bench_close is None:
        raise ValueError(f"Unable to price {bench_sym}.")

    initial_bench_shares = _bench_buy_qty(initial, first_bench_close)
    initial_cost = ib_cost(initial_bench_shares, first_bench_close, False)
    initial_cash = initial - initial_bench_shares * first_bench_close - initial_cost

    bench_shares = initial_bench_shares
    cash = initial_cash
    total_txn_cost = initial_cost
    open_positions, completed, equity_rows = [], [], []
    trade_idx = 0
    skip_max_concurrent = skip_duplicate_symbol = skip_insufficient_capital = 0

    def close_position(pos, close_day, exit_price, exit_reason):
        nonlocal cash, bench_shares, total_txn_cost
        qty = int(pos["_qty"])
        entry_price = float(pos["entry_price"])
        asset_sell_cost = ib_cost(qty, exit_price, True)
        sale_proceeds = qty * exit_price - asset_sell_cost
        bench_close = float(_close_on(sim_prices, bench_sym, close_day) or 0.0)
        rebuy_qty = _bench_buy_qty(sale_proceeds, bench_close)
        rebuy_cost = ib_cost(rebuy_qty, bench_close, False)
        cash += sale_proceeds - rebuy_qty * bench_close - rebuy_cost
        bench_shares += rebuy_qty
        total_txn_cost += asset_sell_cost + rebuy_cost
        direct_cost = (float(pos["_benchmark_sell_cost"]) + float(pos["_asset_buy_cost"])
                       + asset_sell_cost + rebuy_cost)
        gross_pnl = qty * (exit_price - entry_price)
        net_pnl = gross_pnl - direct_cost
        exposure = max(float(pos["_asset_entry_notional"]), 1e-12)
        pos["exit_price"] = round(float(exit_price), 6)
        pos["exit_date"] = str(close_day.date())
        pos["realized_exit_reason"] = exit_reason
        pos["gross_pnl"] = round(gross_pnl, 2)
        pos["pnl"] = round(net_pnl, 2)
        pos["pnl_pct"] = round(net_pnl / exposure * 100.0, 4)
        pos["txn_cost"] = round(direct_cost, 2)
        pos["exit_value"] = round(qty * exit_price, 2)
        pos["benchmark_rebuy_qty"] = round(rebuy_qty, 4)
        completed.append(pos)

    def sweep_idle_cash(bench_close):
        nonlocal cash, bench_shares, total_txn_cost
        if not FULLY_INVESTED_SWEEP or bench_close <= 0 or cash < MIN_SWEEP_CASH:
            return
        qty = _bench_buy_qty(cash, bench_close)
        if qty <= 0:
            return
        buy_cost = ib_cost(qty, bench_close, False)
        cash -= qty * bench_close + buy_cost
        bench_shares += qty
        total_txn_cost += buy_cost

    def try_open_trade(trade, day, bench_close, *, base_ps, max_concurrent):
        nonlocal cash, bench_shares, total_txn_cost
        nonlocal skip_max_concurrent, skip_duplicate_symbol, skip_insufficient_capital
        if len(open_positions) >= max_concurrent:
            skip_max_concurrent += 1
            return False
        if any(pos["symbol"] == trade["symbol"] for pos in open_positions):
            skip_duplicate_symbol += 1
            return False
        position_size = base_ps
        marked_open_value = sum(
            int(pos["_qty"]) * float(_close_on(sim_prices, pos["symbol"], day) or pos["entry_price"])
            for pos in open_positions)
        current_equity = bench_shares * bench_close + marked_open_value + cash
        desired_allocation = current_equity * position_size
        entry_price = float(trade["entry_price"])
        if entry_price <= 0 or desired_allocation < entry_price:
            skip_insufficient_capital += 1
            return False
        cash_contribution = min(max(cash, 0.0), desired_allocation)
        shortfall = desired_allocation - cash_contribution
        if shortfall > 0:
            desired_sell = shortfall / bench_close if FRACTIONAL_BENCHMARK else int(shortfall / bench_close)
            benchmark_sell_qty = min(desired_sell, bench_shares)
        else:
            benchmark_sell_qty = 0.0
        if cash_contribution + benchmark_sell_qty * bench_close < entry_price:
            skip_insufficient_capital += 1
            return False
        benchmark_sell_cost = ib_cost(benchmark_sell_qty, bench_close, True) if benchmark_sell_qty > 0 else 0.0
        available_for_asset = cash_contribution + benchmark_sell_qty * bench_close - benchmark_sell_cost
        asset_qty = _affordable_buy_qty(available_for_asset, entry_price)
        if asset_qty < 1:
            skip_insufficient_capital += 1
            return False
        asset_buy_cost = ib_cost(asset_qty, entry_price, False)
        asset_cash_needed = asset_qty * entry_price + asset_buy_cost
        if asset_cash_needed > available_for_asset + 1e-9:
            skip_insufficient_capital += 1
            return False
        bench_shares -= benchmark_sell_qty
        cash += available_for_asset - asset_cash_needed - cash_contribution
        total_txn_cost += benchmark_sell_cost + asset_buy_cost
        open_positions.append({
            **trade, "_qty": asset_qty, "_position_size_pct": position_size,
            "_asset_entry_notional": round(asset_qty * entry_price, 2),
            "_equity_at_entry": round(current_equity, 2),
            "invested_frac_pct": round(asset_qty * entry_price / max(current_equity, 1e-12) * 100.0, 4),
            "_benchmark_sell_cost": round(benchmark_sell_cost, 2),
            "_asset_buy_cost": round(asset_buy_cost, 2),
            "_entry_ts": trade["_entry_ts"], "_exit_ts": trade["_exit_ts"],
        })
        return True

    for day in calendar:
        base_ps = float(policy.get("position_size_pct", 0.10))
        max_concurrent = int(policy.get("max_concurrent", 10))
        bench_close = _close_on(sim_prices, bench_sym, day)
        if bench_close is None:
            continue
        # Close trades whose planned exit is known by the current day.
        still_open = []
        for pos in open_positions:
            if pos["_exit_ts"] <= day:
                exit_reason = str(pos.get("exit_reason", "strategy_exit"))
                if end_date is not None and exit_reason == "end_of_window":
                    exit_reason = "evaluation_end_liquidation"
                close_position(pos, day, float(pos["exit_price"]), exit_reason)
            else:
                still_open.append(pos)
        open_positions = still_open
        # Open candidates on/after their entry day (FIFO); >4-day gaps are dropped.
        while trade_idx < len(all_trades):
            trade = all_trades[trade_idx]
            if trade["_entry_ts"] > day:
                break
            trade_idx += 1
            if trade["_entry_ts"] < day:
                if (day - trade["_entry_ts"]).days > 4:
                    continue
                trade["_entry_ts"] = day
                trade["entry_date"] = str(day.date())
            try_open_trade(trade, day, float(bench_close), base_ps=base_ps, max_concurrent=max_concurrent)
        sweep_idle_cash(float(bench_close))
        open_value = sum(
            int(pos["_qty"]) * float(_close_on(sim_prices, pos["symbol"], day) or pos["entry_price"])
            for pos in open_positions)
        equity = bench_shares * bench_close + open_value + cash
        passive = initial_bench_shares * bench_close + initial_cash
        equity_rows.append({
            "date": str(day.date()), "equity": round(equity, 2),
            "benchmark_equity": round(passive, 2), "cash": round(cash, 2),
            "benchmark_shares": round(bench_shares, 4), "open_positions": len(open_positions),
        })

    # Forced liquidation at the evaluation horizon.
    if open_positions:
        for pos in list(open_positions):
            forced_price = _close_on(sim_prices, pos["symbol"], last_day)
            if forced_price is None:
                forced_price = float(pos["entry_price"])
            close_position(pos, last_day, float(forced_price), "evaluation_end_liquidation")
        open_positions = []

    final_equity = bench_shares * last_bench_close + cash
    final_passive = initial_bench_shares * last_bench_close + initial_cash
    if equity_rows:
        equity_rows[-1].update({
            "equity": round(final_equity, 2), "benchmark_equity": round(final_passive, 2),
            "cash": round(cash, 2), "benchmark_shares": round(bench_shares, 4), "open_positions": 0,
        })
    else:
        equity_rows.append({
            "date": str(last_day.date()), "equity": round(final_equity, 2),
            "benchmark_equity": round(final_passive, 2), "cash": round(cash, 2),
            "benchmark_shares": round(bench_shares, 4), "open_positions": 0,
        })

    equity_df = pd.DataFrame(equity_rows)
    trade_df = pd.DataFrame(completed)
    equity_values = equity_df["equity"].astype(float).to_numpy()
    peaks = np.maximum.accumulate(equity_values)
    drawdowns = np.where(peaks > 0, equity_values / peaks - 1.0, 0.0)
    max_dd = float(np.min(drawdowns) * 100.0) if len(drawdowns) else 0.0
    adv = _calc_advanced_metrics(equity_df["equity"])
    bench_adv = _calc_advanced_metrics(equity_df["benchmark_equity"])

    if trade_df.empty:
        gross_trade_pnl = net_trade_pnl = trade_txn_cost = 0.0
        win_rate = avg_pnl = avg_gross_pnl = 0.0
        position_sizes = np.asarray([], dtype=float)
    else:
        gross_trade_pnl = float(trade_df["gross_pnl"].sum())
        net_trade_pnl = float(trade_df["pnl"].sum())
        trade_txn_cost = float(trade_df["txn_cost"].sum())
        win_rate = float((trade_df["pnl"] > 0).mean() * 100.0)
        avg_pnl = float(trade_df["pnl"].mean())
        avg_gross_pnl = float(trade_df["gross_pnl"].mean())
        position_sizes = trade_df["_position_size_pct"].astype(float).to_numpy()

    stats = {
        "initial": round(initial, 2), "final": round(final_equity, 2),
        "total_return": round((final_equity / initial - 1.0) * 100.0, 4),
        "benchmark_return": round((final_passive / initial - 1.0) * 100.0, 4),
        "excess_return": round((final_equity - final_passive) / initial * 100.0, 4),
        "max_dd": round(max_dd, 4),
        "sharpe": round(adv["sharpe"], 4), "sortino": round(adv["sortino"], 4),
        "benchmark_sharpe": round(bench_adv["sharpe"], 4), "benchmark_sortino": round(bench_adv["sortino"], 4),
        "n_trades": int(len(trade_df)), "win_rate": round(win_rate, 4),
        "avg_pnl": round(avg_pnl, 4), "avg_gross_pnl": round(avg_gross_pnl, 4),
        "gross_trade_pnl": round(gross_trade_pnl, 2), "net_trade_pnl": round(net_trade_pnl, 2),
        "total_txn_cost": round(total_txn_cost, 2), "trade_txn_cost": round(trade_txn_cost, 2),
        "avg_position_size": round(float(position_sizes.mean() * 100.0), 4) if len(position_sizes) else 0.0,
        "median_position_size": round(float(np.median(position_sizes) * 100.0), 4) if len(position_sizes) else 0.0,
        "min_position_size": round(float(position_sizes.min() * 100.0), 4) if len(position_sizes) else 0.0,
        "max_position_size": round(float(position_sizes.max() * 100.0), 4) if len(position_sizes) else 0.0,
        "start_date": str(first_day.date()), "end_date": str(last_day.date()),
        "n_equity_days": int(len(equity_df)),
        "skip_max_concurrent": skip_max_concurrent, "skip_duplicate_symbol": skip_duplicate_symbol,
        "skip_insufficient_capital": skip_insufficient_capital,
    }
    return trade_df, equity_df, stats


def daily_equity_sharpe(equity_df):
    """Annualized Sharpe from portfolio daily equity returns (CEM objective input)."""
    if equity_df.empty or len(equity_df) < MIN_DAILY_RETURNS_FOR_SHARPE + 1:
        return None
    daily_returns = equity_df["equity"].astype(float).pct_change().dropna()
    if len(daily_returns) < MIN_DAILY_RETURNS_FOR_SHARPE:
        return None
    std = float(daily_returns.std(ddof=1))
    if not np.isfinite(std) or std <= 1e-12:
        return 0.0
    sharpe = float(daily_returns.mean() / std * math.sqrt(252.0))
    return sharpe if np.isfinite(sharpe) else None


def cem_reward(trades, equity_df, stats):
    """Baseline objective: daily-equity Sharpe − DD_PENALTY·|max drawdown %|."""
    if trades.empty or stats["n_trades"] < MIN_TRADES_FOR_REWARD:
        return INVALID_SCORE
    sharpe = daily_equity_sharpe(equity_df)
    if sharpe is None:
        return INVALID_SCORE
    return float(sharpe - DD_PENALTY * abs(float(stats["max_dd"])))


def _cem_fit_policy(fit_df, prices, probs, *, bench_sym, fit_cutoff, fit_eval_end,
                    n_iter, pop, seed, phase_tag):
    """Fit one CEM policy on a label-complete information set; return (policy, score)."""
    assert_rows_completed_before(fit_df, fit_cutoff, context=phase_tag)
    rng = np.random.default_rng(seed)
    names = list(PORTFOLIO_BOUNDS.keys())
    dim = len(names)
    elite_count = max(2, int(pop * CEM_ELITE_FRAC))
    mean = np.array([PORT_DEFAULT[name] for name in names], dtype=float)
    std = np.array([(PORTFOLIO_BOUNDS[name][1] - PORTFOLIO_BOUNDS[name][0]) / 4.0 for name in names], dtype=float)
    fit_start, _ = _frame_bounds(fit_df)
    best_score = -np.inf
    best_policy = None
    for iteration in range(n_iter):
        samples = rng.normal(mean, std, size=(pop, dim))
        policies = [port_policy_from_vec(sample) for sample in samples]
        scores = []
        for policy in policies:
            trades, equity, stats = sim_opp_cost(
                fit_df, prices, probs, policy, bench_sym=bench_sym, initial=INITIAL_CAPITAL,
                start_date=fit_start, end_date=fit_eval_end)
            scores.append(cem_reward(trades, equity, stats))
        score_array = np.asarray(scores, dtype=float)
        elite_idx = np.argsort(score_array)[-elite_count:]
        elite = samples[elite_idx]
        mean = elite.mean(axis=0)
        std = elite.std(axis=0) + 1e-4
        iteration_best_idx = int(np.argmax(score_array))
        iteration_best_score = float(score_array[iteration_best_idx])
        if iteration_best_score > best_score:
            best_score = iteration_best_score
            best_policy = policies[iteration_best_idx]
        print(f"    {bench_sym}|{phase_tag} iter {iteration + 1}/{n_iter}  "
              f"best={iteration_best_score:+.3f}  global={best_score:+.3f}", flush=True)
    if best_policy is None or best_score <= INVALID_SCORE / 2:
        raise RuntimeError(f"No valid CEM policy for {bench_sym} in {phase_tag}.")
    return best_policy, float(best_score)


def cem_search_baseline(df, prices, probs, *, bench_sym, train_fit_cutoff,
                        n_iter=CEM_ITERS, pop=CEM_POP, seed=CEM_BASE_SEED):
    """Baseline CEM: a single fit on all label-complete train rows (t_e < cutoff)."""
    train_fit_cutoff = as_utc_day(train_fit_cutoff)
    final_fit_df = rows_completed_before(df, train_fit_cutoff)
    assert_rows_completed_before(final_fit_df, train_fit_cutoff, context="final train fit")
    final_fit_end = train_fit_cutoff - pd.Timedelta(days=1)
    return _cem_fit_policy(final_fit_df, prices, probs, bench_sym=bench_sym,
                           fit_cutoff=train_fit_cutoff, fit_eval_end=final_fit_end,
                           n_iter=n_iter, pop=pop, seed=seed, phase_tag="TrainFull")


print("portfolio simulator + CEM search ready (Baseline arm)")

## 3. The original data pipeline (documentation only)

The artifacts this notebook consumes are the end product of the project's ingestion
pipeline (the `ingest/` package in the full research repository — **not required to run
this notebook**). None of these steps are re-run here: no Polymarket download, no Gemini
calls, no cleaning re-run, no database connection.

| Stage | What it does |
|---|---|
| **1. Download Polymarket questions** (`ingest/scanner.py`) | Polymarket markets are fetched from the **Gamma API** `/events` endpoint — question text, market ids, tags, end dates — restricted to financial / macro / geopolitical categories resolving 5–60 days out. Hourly market-probability history and daily equity bars are downloaded alongside. |
| **2. Deterministic filtering & cleaning** (`ingest/prefilter.py`) | A free regex/keyword gate culls structural noise **before** any paid API call: sports/entertainment/pop-culture questions, cryptocurrency price targets, raw stock/index price-level questions, operational-metric threshold ladders. |
| **3. Duplicate handling** (`ingest/dedup.py`) | Polymarket publishes macro events as bracket ladders of near-identical questions ("Fed cuts 25 / 50 / 75+ bps…", "CPI 2.4 / 2.5 / 2.6 %…"). The rungs collapse to a single economic event so one catalyst cannot multi-count. |
| **4. Gemini catalyst gate** (`ingest/evaluator.py`, `ingest/gemini_client.py`) | Each surviving question is sent to **Gemini**, which decides whether it is a real, tradable financial catalyst for US equities. |
| **5. Relevance scoring & asset mapping** (`ingest/world.py`) | A second Gemini pass maps each question to specific **US stocks and ETFs**, each with a causal channel and a *connection strength* (relevance ∈ [0, 1]). The backtest keeps pairs with strength > 0.5. |
| **6. Polarity assignment** (`ingest/label_polarity.py`, `core/polarity.py`) | Every (question, symbol) pair gets a direction: does rising P(YES) mean *good* or *bad* news for the asset? Resolution order: manual **override** → committed **LLM label** → **regex** fallback. Labels are committed in `data/polarity_labels.json`. The book is long-only: bearish-YES pairs trade the flipped path `1 − P(YES)`; pairs with no clean side (polarity 0) are skipped. |
| **7. Final PKL / Parquet artifacts** (`ingest/artifacts.py`, `ingest/clean_candidates.py`) | Point-in-time candidate features (**T0** = market creation, **Tθ** = probability-threshold crossing, **Te** = resolution) are computed per (market, symbol) and written to the committed artifacts below. |

**Committed artifacts loaded by this notebook** (relative paths only):

- `data/candidates_audit_clean.parquet` — the audit-cleaned, primary-eligible candidate set (one row per market × symbol, with features and the chronological split)
- `data/prices.pkl` — `{symbol: [(ts, high, low, close), …]}` daily bars, including SPY and QQQ
- `data/probs.pkl` — `{market_id: [(ts, P(YES)), …]}` daily probability paths
- `data/polarity_labels.json` — committed per-pair polarity labels (loaded automatically by `core/polarity.py`)

In [ ]:
for p in (CANDIDATES_PATH, PRICES_PATH, PROBS_PATH, POLARITY_PATH):
    assert p.exists(), f"Missing committed artifact: {p}"

candidates = pd.read_parquet(CANDIDATES_PATH)
with open(PRICES_PATH, "rb") as f:
    prices = pickle.load(f)
with open(PROBS_PATH, "rb") as f:
    probs = pickle.load(f)

# The exact universe the strategy trades: primary-eligible rows with Gemini
# connection strength > 0.5 (the same filter the CEM search applies below).
eligible = candidates
if "cem_eligible" in eligible.columns:
    eligible = eligible[eligible["cem_eligible"].fillna(False).astype(bool)]
eligible = eligible[eligible[RELEVANCE_COL].astype(float) > 0.5].copy()

t_theta = pd.to_datetime(eligible["t_theta"], utc=True)
print(f"candidate rows (raw parquet)  : {len(candidates)}")
print(f"candidate rows (CEM universe) : {len(eligible)}")
print(f"unique Polymarket questions   : {eligible['question'].nunique()}")
print(f"unique question→asset pairs   : {len(eligible[['question', 'symbol']].drop_duplicates())}")
print(f"unique symbols                : {eligible['symbol'].nunique()}")
print(f"price series loaded           : {len(prices)} symbols")
print(f"probability paths loaded      : {len(probs)} markets")
print(f"candidate window (t_theta)    : {t_theta.min().date()} → {t_theta.max().date()}")

### The complete final mapping — every question, every mapped asset

The table below is the **complete** question → stock/ETF mapping the standard CEM
strategy trades from — one row per question–symbol pair, with the Gemini connection
strength (`relevance`) and the resolved signal polarity
(`+1` = rising P(YES) is bullish for the symbol, `−1` = the strategy trades the flipped
path `1 − P(YES)`, `0` = no clean side, the pair is skipped by the kernel).

The full table is displayed below (no preview truncation) and exported to
`notebooks/outputs/question_to_assets_mapping.csv`.

In [ ]:
# Uses the inlined resolve_polarity (§2.1) — no project import.
mapping = (
    eligible[["market_id", "question", "symbol", "feat_connection_strength"]]
    .drop_duplicates(subset=["question", "symbol"])
    .rename(columns={"feat_connection_strength": "relevance"})
    .sort_values(["question", "symbol"])
    .reset_index(drop=True)
)
resolved = [resolve_polarity(q, s) for q, s in zip(mapping["question"], mapping["symbol"])]
mapping["polarity"] = [r[0] for r in resolved]
mapping["polarity_source"] = [r[1] for r in resolved]

mapping_csv = OUT_DIR / "question_to_assets_mapping.csv"
mapping.to_csv(mapping_csv, index=False, encoding="utf-8")
print(f"exported {len(mapping)} question→asset rows "
      f"({mapping['question'].nunique()} unique questions) → {mapping_csv}\n")

# Display the COMPLETE mapping — every question and every asset mapped to it.
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(mapping)

## 4. The simple standard CEM strategy

This runs the **Baseline** arm of the project's experiment matrix — the plain
cross-entropy-method strategy with **none** of the treatment-ladder techniques:

- ❌ no T1 realized-friction penalty, ❌ no T2 walk-forward refits, ❌ no T3 half-Kelly
  sizing, ❌ no T4 geo-priority allocation, ❌ no PPO, ❌ no ablation ladder;
- ✅ one CEM fit on label-complete training data, **frozen**, then evaluated once
  out-of-sample.

The cell below calls the inlined engine (§2) directly, **in-process** — no subprocess,
no CLI, no project import. It uses the same CEM fitting, entry/exit rules, position
sizing, portfolio capacity, transaction costs, benchmark-capital handling, and train/test
chronology defined in §2.3.

**CEM fitting** (`cem_search_baseline`): 6 iterations × population 20, elite fraction
25 %, base seed 42 (QQQ adds a fixed offset of 10 000, so each benchmark searches from its
own fixed initial population). Fitness of a candidate policy = Sharpe of the simulated
**daily portfolio equity** − 0.30 × |max drawdown %|; policies producing fewer than
3 trades or fewer than 20 daily returns are invalid. The fit may only use candidates
whose outcome was complete before the OOS boundary (`t_e` < 2026-01-01), and every
fit simulation truncates price/probability paths at its cutoff — label-complete,
leakage-safe training.

**Policy parameters and CEM bounds** (`PORTFOLIO_BOUNDS` in §2.3):

| Parameter | Bounds | Role |
|---|---|---|
| `enter_floor` | 0.55 – 0.80 | probability level an entry signal must hold |
| `enter_strong` | 0.60 – 0.85 | immediate-entry probability level |
| `hold_days` | 1 – 5 | consecutive days ≥ `enter_floor` required when below `enter_strong` |
| `max_prob_surge` | 0.20 – 0.80 | entry veto if the probability already surged since T0 |
| `max_price_runup` | 0.02 – 0.20 | entry veto if the equity already ran up since T0 |
| `theta_out` | 0.45 – 0.60 | exit when the effective probability collapses below this |
| `atr_mult` | 1.5 – 4.0 | trailing ATR stop width |
| `lock_activate` | 0.02 – 0.10 | profit level that arms the profit-lock floor |
| `position_size_pct` | 0.06 – 0.12 | fraction of portfolio equity per position |
| `max_concurrent` | 8 – 12 | portfolio capacity (simultaneously open positions) |

**Entry rule** (`_scan` in §2.2): starting the day the market's polarity-adjusted
probability first crosses θ (**Tθ**), enter when it reaches `enter_strong` immediately, or
holds ≥ `enter_floor` for `hold_days` consecutive days — subject to the `max_prob_surge` /
`max_price_runup` vetoes. Long-only, whole shares, one position per symbol.

**Exit rules** (first to trigger wins): trailing ATR stop (`atr_mult`), profit lock
(after a `lock_activate` gain, exit if price falls back to the locked floor),
probability collapse below `theta_out`, forced exit one trading day before market
resolution, or end-of-window liquidation.

**Position sizing & portfolio capacity**: fixed `position_size_pct` of current portfolio
equity per position (the standard arm uses no Kelly), capped at `max_concurrent` open
positions; signals arriving while the roster is full are skipped in FIFO order.

**Transaction costs & benchmark-capital handling**: capital starts at \$100,000 and is
**fully invested at all times** — idle cash is swept into the benchmark (fractional
SPY/QQQ shares; sweeps under \$100 are skipped). Funding an event position sells
benchmark shares and closing it re-buys them, so every round trip pays **four** cost
legs (benchmark sell → asset buy → asset sell → benchmark re-buy), each charged
IB-style: `max($0.35, $0.0035/share)` commission capped at 1 % of trade value, SEC fee
on sells, plus 5 bp slippage.

**Training and backtest chronology**: candidate features are point-in-time
(T0 / Tθ / Te). The CEM fit sees only outcomes completed before **2026-01-01**; the
fitted policy is then frozen and run once over the out-of-sample window
**2026-01-01 → 2026-06-12**, marked to market daily against the benchmark bought and
held with the same initial capital.

The run covers **all events used by the existing standard CEM experiment**: the full
audit-clean candidate universe loaded above, on both benchmarks.

In [ ]:
# Run the standard CEM strategy end-to-end, in-process, using ONLY the engine
# defined in §2. No subprocess, no CLI, no project import.

# 1) Build the exact CEM candidate universe (same filters the driver applies).
df = candidates.copy()
if "cem_eligible" in df.columns:
    df = df.loc[df["cem_eligible"].fillna(False).astype(bool)].copy()
df = df[df[RELEVANCE_COL].astype(float) > 0.5].copy()
if "split" in df.columns:
    df["split"] = df["split"].astype(str).str.lower().str.strip().replace({"val": "test"})
df["t_theta"] = pd.to_datetime(df["t_theta"], utc=True)
df["t_e"] = pd.to_datetime(df["t_e"], utc=True)
assert not (df["t_e"] < df["t_theta"]).any(), "found candidate rows with t_e earlier than t_theta"

# 2) Fixed chronological split. The OOS boundary is 2026-01-01 and is never recomputed.
OOS_START = pd.Timestamp("2026-01-01", tz="UTC")
max_t = as_utc_day(df["t_theta"].max()) + pd.Timedelta(days=1)
OOS_END = max_t - pd.Timedelta(days=1)
train_df = rows_completed_before(df, OOS_START)          # t_e < 2026-01-01 (label-complete)
train_eval_end = OOS_START - pd.Timedelta(days=1)
oos_df = df[(df["t_theta"] >= OOS_START) & (df["t_theta"] <= OOS_END)].copy()

print(f"  universe candidates           = {len(df)}")
print(f"  train samples (t_e < {OOS_START.date()}) = {len(train_df)}")
print(f"  OOS test samples              = {len(oos_df)}")
print(f"  OOS window                    = {OOS_START.date()} → {OOS_END.date()}\n")

# Fresh caches so re-running the cell starts clean.
clear_kernel_caches()
_CLOSE_CACHE.clear()
_PATH_CUTOFF_CACHE.clear()

# 3) For each benchmark: fit the CEM policy on the train window, freeze it, then
#    run the frozen policy once over the out-of-sample window. QQQ uses a fixed
#    seed offset so each benchmark searches from its own fixed initial population.
results_rows, equity_logs, trade_logs, fitted = [], {}, {}, {}
for bench in ("SPY", "QQQ"):
    seed = CEM_BASE_SEED + BENCHMARK_SEED_OFFSET[bench]
    print(f"[Train CEM search — {bench}]  seed={seed}", flush=True)
    policy, train_score = cem_search_baseline(
        df, prices, probs, bench_sym=bench, train_fit_cutoff=OOS_START, seed=seed)

    # Train-window diagnostic (IN-SAMPLE for the frozen baseline).
    tr_trades, tr_equity, tr_stats = sim_opp_cost(
        train_df, prices, probs, policy, bench_sym=bench,
        start_date=as_utc_day(train_df["t_theta"].min()), end_date=train_eval_end)

    # Out-of-sample: the frozen policy, evaluated once, never consulted in the fit.
    print(f"[OOS sim — {bench}]", flush=True)
    oos_trades, oos_equity, oos_stats = sim_opp_cost(
        oos_df, prices, probs, policy, bench_sym=bench, start_date=OOS_START, end_date=OOS_END)

    fitted[bench] = policy
    equity_logs[(bench, "train")] = tr_equity
    equity_logs[(bench, "test")] = oos_equity
    trade_logs[bench] = oos_trades
    results_rows.append({
        "benchmark": bench,
        "test_start_date": oos_stats["start_date"], "test_end_date": oos_stats["end_date"],
        "test_return_pct": oos_stats["total_return"],
        "test_benchmark_return_pct": oos_stats["benchmark_return"],
        "test_excess_return_pct": oos_stats["excess_return"],
        "test_sharpe": oos_stats["sharpe"], "test_benchmark_sharpe": oos_stats["benchmark_sharpe"],
        "test_sortino": oos_stats["sortino"], "test_max_dd_pct": oos_stats["max_dd"],
        "test_trades": oos_stats["n_trades"], "test_win_rate_pct": oos_stats["win_rate"],
        "test_total_txn_cost": oos_stats["total_txn_cost"], "test_trade_txn_cost": oos_stats["trade_txn_cost"],
        "test_avg_position_size_pct": oos_stats["avg_position_size"],
        "train_return_pct": tr_stats["total_return"],
        "train_benchmark_return_pct": tr_stats["benchmark_return"],
        "train_excess_return_pct": tr_stats["excess_return"],
        "train_sharpe": tr_stats["sharpe"], "train_max_dd_pct": tr_stats["max_dd"],
        "train_trades": tr_stats["n_trades"], "train_win_rate_pct": tr_stats["win_rate"],
    })
    print(f"  -> TRAIN (in-sample) = {tr_stats['total_return']:+.2f}%  "
          f"B&H {tr_stats['benchmark_return']:+.2f}%  trades={tr_stats['n_trades']}")
    print(f"  -> TEST  (OOS)       = {oos_stats['total_return']:+.2f}%  "
          f"B&H {oos_stats['benchmark_return']:+.2f}%  excess {oos_stats['excess_return']:+.2f}%  "
          f"max_dd {oos_stats['max_dd']:+.2f}%  sharpe {oos_stats['sharpe']:.2f} "
          f"(B&H {oos_stats['benchmark_sharpe']:.2f})  trades={oos_stats['n_trades']}  "
          f"win%={oos_stats['win_rate']:.1f}%\n", flush=True)

results = pd.DataFrame(results_rows)
print("Baseline CEM finished OK.")

## 5. Results — SPY and QQQ

The run above produced everything in memory. The cells below summarize **both stages** —
the train window and the out-of-sample test window — and plot the CEM portfolio against
each benchmark.

> **Reading the two stages honestly:** the frozen baseline policy was *fitted* on the
> train window (candidates with `t_e` < 2026-01-01), so the train equity curve is
> **in-sample** — a diagnostic, not evidence of edge. The test curve (2026-01-02 →
> 2026-06-12) is the genuine out-of-sample result: the policy was frozen before the
> boundary and the window was never consulted during fitting.

In [ ]:
rows = []
for bench in ("SPY", "QQQ"):
    r = results.loc[results["benchmark"] == bench].iloc[0]
    eq = equity_logs[(bench, "test")]
    rows.append({
        "benchmark": bench,
        "oos_start": r["test_start_date"],
        "oos_end": r["test_end_date"],
        "portfolio_return_pct": float(r["test_return_pct"]),
        "benchmark_return_pct": float(r["test_benchmark_return_pct"]),
        "excess_return_pct": float(r["test_excess_return_pct"]),
        "sharpe": float(r["test_sharpe"]),
        "benchmark_sharpe": float(r["test_benchmark_sharpe"]),
        "max_drawdown_pct": float(r["test_max_dd_pct"]),
        "terminal_equity": float(eq["equity"].iloc[-1]),
        "n_trades": int(r["test_trades"]),
        "win_rate_pct": float(r["test_win_rate_pct"]),
        "total_txn_costs": float(r["test_total_txn_cost"]),
        "trade_txn_costs": float(r["test_trade_txn_cost"]),
        # Train window (IN-SAMPLE for the frozen baseline — diagnostic only):
        "train_return_pct": float(r["train_return_pct"]),
        "train_benchmark_return_pct": float(r["train_benchmark_return_pct"]),
        "train_excess_return_pct": float(r["train_excess_return_pct"]),
        "train_n_trades": int(r["train_trades"]),
        "train_terminal_equity": float(equity_logs[(bench, "train")]["equity"].iloc[-1]),
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "standard_cem_summary.csv", index=False, encoding="utf-8")
print(f"summary exported → {OUT_DIR / 'standard_cem_summary.csv'}")
display(summary.round(4))

In [ ]:
# The fitted CEM policy parameters (the frozen policy each benchmark trades OOS).
params_path = OUT_DIR / "cem_fitted_parameters.json"
params_path.write_text(json.dumps(fitted, indent=2, sort_keys=True), encoding="utf-8")
print(f"fitted CEM parameters exported → {params_path}")
display(pd.DataFrame(fitted).rename_axis("parameter"))

In [ ]:
def plot_equity(bench: str, stage: str, save_path: Path) -> None:
    eq = equity_logs[(bench, stage)]
    dates = pd.to_datetime(eq["date"])
    port_ret = (eq["equity"].iloc[-1] / eq["equity"].iloc[0] - 1) * 100
    bench_ret = (eq["benchmark_equity"].iloc[-1] / eq["benchmark_equity"].iloc[0] - 1) * 100
    if stage == "train":
        stage_label = (f"TRAIN {dates.iloc[0].date()} → {dates.iloc[-1].date()} "
                       "(IN-SAMPLE: the policy was fitted here)")
    else:
        stage_label = (f"TEST {dates.iloc[0].date()} → {dates.iloc[-1].date()} "
                       "(OUT-OF-SAMPLE, untouched by the fit)")
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.plot(dates, eq["equity"], label="Standard CEM portfolio",
            color="#1f77b4", linewidth=2)
    ax.plot(dates, eq["benchmark_equity"], label=f"{bench} buy & hold",
            color="#2ca02c", linestyle="--", linewidth=2)
    ax.set_title(
        f"Standard CEM vs {bench} — {stage_label}\n"
        f"return {port_ret:+.2f}% vs {bench_ret:+.2f}%   |   "
        f"terminal equity ${eq['equity'].iloc[-1]:,.0f}",
        fontsize=11,
    )
    ax.set_ylabel("Equity ($)")
    ax.set_xlabel("Date")
    ax.legend(loc="upper left")
    ax.grid(True, linestyle="--", alpha=0.6)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    print(f"figure saved → {save_path}")
    plt.show()


plot_equity("SPY", "train", OUT_DIR / "cem_vs_spy_equity_train.png")
plot_equity("SPY", "test", OUT_DIR / "cem_vs_spy_equity.png")

In [ ]:
plot_equity("QQQ", "train", OUT_DIR / "cem_vs_qqq_equity_train.png")
plot_equity("QQQ", "test", OUT_DIR / "cem_vs_qqq_equity.png")

## 6. Output files

All practical outputs are collected in **`notebooks/outputs/`**:

| File | Contents |
|---|---|
| `question_to_assets_mapping.csv` | the complete question → stock/ETF mapping (with relevance and polarity) |
| `spy_trades.csv` / `qqq_trades.csv` | every out-of-sample trade (entry/exit dates, prices, shares, net P&L, exit reason, all four cost legs) |
| `spy_daily_equity.csv` / `qqq_daily_equity.csv` | daily portfolio equity vs benchmark buy-&-hold equity |
| `standard_cem_summary.csv` | portfolio return, benchmark return, excess return, Sharpe, max drawdown, terminal equity, number of trades, transaction costs (test window, plus in-sample train diagnostics) |
| `cem_fitted_parameters.json` | the fitted (frozen) CEM policy per benchmark |
| `cem_vs_spy_equity.png` / `cem_vs_qqq_equity.png` | the two out-of-sample comparison figures |
| `cem_vs_spy_equity_train.png` / `cem_vs_qqq_equity_train.png` | the train-window comparison figures (in-sample for the frozen baseline) |

Everything above is produced by this notebook alone, from the four committed `data/`
artifacts — there is no external audit-trail directory to consult.

In [ ]:
# Write the practical outputs straight from the in-memory results (no runs/ dir).
trade_logs["SPY"].to_csv(OUT_DIR / "spy_trades.csv", index=False)
trade_logs["QQQ"].to_csv(OUT_DIR / "qqq_trades.csv", index=False)
equity_logs[("SPY", "test")].to_csv(OUT_DIR / "spy_daily_equity.csv", index=False)
equity_logs[("QQQ", "test")].to_csv(OUT_DIR / "qqq_daily_equity.csv", index=False)

print(f"Practical outputs in {OUT_DIR}:\n")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:38s} {p.stat().st_size:>10,} bytes")

## 7. Verification

The standard CEM experiment is deterministic for a given seed. The cell below checks
this run against the project's canonical seed-42 standard-CEM results (the `Baseline`
rows of `runs/icaif_base_vs_all_seed42`, i.e. the numbers behind the paper's
seed-42 matrix), and that every promised output file exists.

In [ ]:
CANONICAL = {
    "SPY": {"portfolio_return_pct": 30.2742, "benchmark_return_pct": 8.5199,
            "excess_return_pct": 21.7543, "sharpe": 3.2975,
            "max_drawdown_pct": -6.6651, "n_trades": 225,
            "train_return_pct": 53.2631, "train_n_trades": 164},
    "QQQ": {"portfolio_return_pct": 27.3641, "benchmark_return_pct": 17.5912,
            "excess_return_pct": 9.7729, "sharpe": 2.9321,
            "max_drawdown_pct": -6.3585, "n_trades": 223,
            "train_return_pct": 59.7443, "train_n_trades": 155},
}

required_outputs = [
    "question_to_assets_mapping.csv",
    "spy_trades.csv", "qqq_trades.csv",
    "spy_daily_equity.csv", "qqq_daily_equity.csv",
    "standard_cem_summary.csv", "cem_fitted_parameters.json",
    "cem_vs_spy_equity.png", "cem_vs_qqq_equity.png",
    "cem_vs_spy_equity_train.png", "cem_vs_qqq_equity_train.png",
]
missing = [n for n in required_outputs if not (OUT_DIR / n).exists()]
assert not missing, f"missing output files: {missing}"
assert set(results["benchmark"]) == {"SPY", "QQQ"}
assert len(mapping) == mapping[["question", "symbol"]].drop_duplicates().shape[0]
assert pd.read_csv(OUT_DIR / "question_to_assets_mapping.csv").shape[0] == len(mapping)

report, all_match = [], True
for bench, ref in CANONICAL.items():
    row = summary.loc[summary["benchmark"] == bench].iloc[0]
    for metric, expected in ref.items():
        got = float(row[metric])
        tol = 0.5 if metric.endswith("n_trades") else 1e-3
        ok = abs(got - float(expected)) <= tol
        all_match &= ok
        report.append({"benchmark": bench, "metric": metric,
                       "this_run": got, "canonical_run": expected, "match": ok})
display(pd.DataFrame(report))

print(f"output files                 : all {len(required_outputs)} present")
print(f"benchmarks                   : {sorted(set(results['benchmark']))}")
if all_match:
    print("RESULT MATCH                 : this self-contained notebook reproduces the "
          "canonical standard CEM experiment exactly.")
else:
    print("RESULT MATCH                 : DIFFERS from the canonical run — likely "
          "library-version drift; inspect the table above.")